# Chess tokenizer exploration

First pass at a compositional SAN/movetext tokenizer for the Lumbras PGN splits.


In [ ]:
import re
from dataclasses import dataclass
from typing import Iterable

# Chess-aware SAN tokenizer for PGN movetext.
# Goal: expressive, compositional tokens without one token per possible move/position.
#
# Examples:
#   Nbd7      -> PIECE_N SRC_FILE_b TO_d7 <EOM>
#   exd8=Q+   -> PAWN SRC_FILE_e CAPTURE TO_d8 PROMO_Q CHECK <EOM>
#   O-O       -> CASTLE_KINGSIDE <EOM>
#   R1e2      -> PIECE_R SRC_RANK_1 TO_e2 <EOM>
#   Qh5#      -> PIECE_Q TO_h5 MATE <EOM>

FILES = "abcdefgh"
RANKS = "12345678"
PIECES = set("KQRBN")
RESULTS = {"1-0": "RESULT_WHITE", "0-1": "RESULT_BLACK", "1/2-1/2": "RESULT_DRAW", "*": "RESULT_UNKNOWN"}

# Strip PGN comments, recursive annotation variations, and common numeric annotation glyphs.
COMMENT_RE = re.compile(r"\{[^}]*\}|;[^\n]*")
NAG_RE = re.compile(r"\$\d+")
MOVE_NUMBER_RE = re.compile(r"^\d+\.(?:\.\.)?$")

SAN_RE = re.compile(
    r"^"
    r"(?P<piece>[KQRBN])?"          # absent means pawn
    r"(?P<src_file>[a-h])?"         # disambiguation or pawn capture source file
    r"(?P<src_rank>[1-8])?"         # disambiguation
    r"(?P<capture>x)?"
    r"(?P<target>[a-h][1-8])"
    r"(?:=(?P<promo>[QRBN]))?"
    r"(?P<suffix>[+#])?"
    r"(?P<annotation>[!?]+)?"
    r"$"
)


def strip_variations(text: str) -> str:
    """Remove parenthesized PGN variations, including nested ones."""
    out = []
    depth = 0
    for ch in text:
        if ch == "(":
            depth += 1
            continue
        if ch == ")" and depth:
            depth -= 1
            continue
        if depth == 0:
            out.append(ch)
    return "".join(out)


def clean_movetext(text: str) -> str:
    """Remove metadata-adjacent PGN clutter while preserving SAN move text."""
    text = COMMENT_RE.sub(" ", text)
    text = strip_variations(text)
    text = NAG_RE.sub(" ", text)
    return text.replace("\n", " ")


def tokenize_san(san: str, add_eom: bool = True) -> list[str]:
    """Tokenize one SAN move into compositional chess tokens.

    <EOM> marks the boundary after a complete move, which makes detokenization
    and legality-check handoff to an external chess environment unambiguous.
    Result tokens do not receive <EOM> because they terminate the game, not a move.
    """
    san = san.strip()
    if not san:
        return []

    # Normalize zeros sometimes used for castling.
    san = san.replace("0", "O") if san in {"0-O", "0-O-O", "0-0", "0-0-0"} else san

    if san in RESULTS:
        return [RESULTS[san]]

    # Castling can carry check/checkmate suffixes.
    if san.startswith("O-O-O"):
        suffix = san[5:]
        tokens = ["CASTLE_QUEENSIDE"]
    elif san.startswith("O-O"):
        suffix = san[3:]
        tokens = ["CASTLE_KINGSIDE"]
    else:
        # Drop common trailing annotations before structural parse.
        san_core = re.sub(r"[!?]+$", "", san)
        match = SAN_RE.match(san_core)
        if not match:
            return ["UNK_SAN", f"RAW_{san}"]

        gd = match.groupdict()
        piece = gd["piece"] or "P"
        tokens = ["PAWN" if piece == "P" else f"PIECE_{piece}"]

        if gd["src_file"]:
            # For pawn captures this is the source file; for pieces it is disambiguation.
            tokens.append(f"SRC_FILE_{gd['src_file']}")
        if gd["src_rank"]:
            tokens.append(f"SRC_RANK_{gd['src_rank']}")
        if gd["capture"]:
            tokens.append("CAPTURE")

        tokens.append(f"TO_{gd['target']}")

        if gd["promo"]:
            tokens.append(f"PROMO_{gd['promo']}")

        suffix = gd["suffix"] or ""

    if "+" in suffix:
        tokens.append("CHECK")
    if "#" in suffix:
        tokens.append("MATE")
    if add_eom:
        tokens.append("<EOM>")

    return tokens


def tokenize_movetext(movetext: str, include_turn_tokens: bool = True) -> list[str]:
    """Tokenize PGN movetext into a flat model-ready token sequence."""
    cleaned = clean_movetext(movetext)
    tokens: list[str] = []
    ply = 0

    for raw in cleaned.split():
        raw = raw.strip()
        if not raw:
            continue

        # Split forms like "1.e4" or "1...Nf6". If a token explicitly
        # uses black-to-move notation, align the local ply counter with black.
        explicit_black_move = bool(re.match(r"^\d+\.\.\.", raw))
        raw = re.sub(r"^\d+\.{1,3}", "", raw)
        if not raw or MOVE_NUMBER_RE.match(raw):
            continue
        if explicit_black_move and ply % 2 == 0:
            ply += 1

        move_tokens = tokenize_san(raw)
        if include_turn_tokens and move_tokens and not move_tokens[0].startswith("RESULT_"):
            tokens.append("WHITE" if ply % 2 == 0 else "BLACK")
            ply += 1
        tokens.extend(move_tokens)
        if move_tokens and move_tokens[0].startswith("RESULT_"):
            break

    return tokens


def pgn_to_movetext(pgn: str) -> str:
    """Drop PGN tag-pair metadata and return only movetext lines."""
    return "\n".join(line for line in pgn.splitlines() if not line.startswith("["))


# Quick sanity checks / examples.
examples = ["Nbd7", "exd8=Q+", "O-O", "O-O-O#", "R1e2", "Qh5#", "1.e4", "1...Nf6"]
for ex in examples:
    print(f"{ex:10s} -> {tokenize_movetext(ex) if ex[0].isdigit() else tokenize_san(ex)}")

sample_movetext = "1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 4. Bxc6 dxc6 5. O-O Nf6 1/2-1/2"
print("\nsample:", tokenize_movetext(sample_movetext))


## Network representation sketch

We will treat each chess move as a fixed-width **move packet** rather than a single opaque move token.

For now:

- `MOVE_SEQ_LEN = 8`
- each row is one full SAN move represented as token ids
- `<EOM>` marks the end of the real move
- `<PAD>` fills the remaining packet positions
- `<PAD>` is id `0`, so it is easy to mask out of losses / legality logic

Examples:

```text
Nbd7     -> [PIECE_N, SRC_FILE_b, TO_d7, <EOM>, <PAD>, <PAD>, <PAD>, <PAD>]
exd8=Q+  -> [PAWN, SRC_FILE_e, CAPTURE, TO_d8, PROMO_Q, CHECK, <EOM>, <PAD>]
O-O      -> [CASTLE_KINGSIDE, <EOM>, <PAD>, <PAD>, <PAD>, <PAD>, <PAD>, <PAD>]
```

Training table shape:

```text
moves_ids: uint16[num_moves, 8]
```

For sequence modeling, we likely form examples as previous move packets predicting the next move packet:

```text
x: int64[batch, context_moves, 8]
y: int64[batch, 8]
```

For a diffusion-style move head, `y` is the denoising target over the 8 packet slots. Each slot predicts a categorical distribution over the vocabulary. `<PAD>` slots after `<EOM>` should be included as structural targets but masked/handled distinctly from legal chess tokens.

Important: this is still only a notation representation. Legal move validation/masking will be handled by an external chess environment later.


In [ ]:
from pathlib import Path
import sys
import numpy as np

# Use the reusable tokenizer module from ../src
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from chessgm.tokenizer import ChessTokenizer, VOCAB

MOVE_SEQ_LEN = 8
tok = ChessTokenizer()

PAD_ID = tok.token_to_id["<PAD>"]
EOM_ID = tok.token_to_id["<EOM>"]

print("vocab_size:", len(VOCAB))
print("PAD_ID:", PAD_ID)
print("EOM_ID:", EOM_ID)

def encode_move_packet(san: str, seq_len: int = MOVE_SEQ_LEN) -> np.ndarray:
    """Encode one SAN move as a fixed-width packet of token ids."""
    encoded = tok.encode_move(san)
    if len(encoded.ids) > seq_len:
        raise ValueError(f"Move {san!r} needs {len(encoded.ids)} ids > seq_len={seq_len}: {encoded.tokens}")
    return np.array(encoded.ids + [PAD_ID] * (seq_len - len(encoded.ids)), dtype=np.uint16)

def decode_move_packet(packet: np.ndarray) -> str:
    """Decode one fixed-width packet back to SAN, ignoring PAD."""
    ids = [int(i) for i in packet if int(i) != PAD_ID]
    return tok.decode_move(ids)

examples = ["e4", "Nbd7", "exd8=Q+", "O-O", "O-O-O#"]
packets = np.stack([encode_move_packet(m) for m in examples])

for move, packet in zip(examples, packets):
    tokens = tok.decode_ids(packet.tolist())
    print(f"{move:8s} -> ids={packet.tolist()} tokens={tokens} decoded={decode_move_packet(packet)}")

print("\npacket table shape:", packets.shape, packets.dtype)


In [ ]:
# Mock model batch construction.
# Suppose a game has N move packets. A training example can be:
#   x = previous context window of move packets
#   y = next move packet

CONTEXT_MOVES = 4
mock_game = ["e4", "e5", "Nf3", "Nc6", "Bb5", "a6", "Bxc6", "dxc6", "O-O"]
game_packets = np.stack([encode_move_packet(m) for m in mock_game])

xs = []
ys = []
for i in range(CONTEXT_MOVES, len(game_packets)):
    xs.append(game_packets[i-CONTEXT_MOVES:i])
    ys.append(game_packets[i])

x = np.stack(xs).astype(np.int64)
y = np.stack(ys).astype(np.int64)

print("x shape:", x.shape, "# [batch, context_moves, move_seq_len]")
print("y shape:", y.shape, "# [batch, move_seq_len]")
print("first target ids:", y[0].tolist())
print("first target tokens:", tok.decode_ids(y[0].tolist()))
print("first target SAN:", decode_move_packet(y[0]))

# Useful masks:
pad_mask = (y == PAD_ID)          # slots that are padding
nonpad_mask = (y != PAD_ID)       # slots that participate in move structure
post_eom_is_pad = y[:, -1] == PAD_ID

print("pad_mask shape:", pad_mask.shape)
print("nonpad slots in first target:", nonpad_mask[0].tolist())
